# Gamerdust - Cuaderno tecnico y funcional

Este cuaderno explica el estado actual de la aplicacion Gamerdust, creada con Laravel, Inertia y React. Esta pensado para entender que hace la app, como se conectan sus piezas, que tablas existen, cual es el flujo del usuario y si actualmente usa IA real o solo logica local.

**Fecha de revision:** 2026-06-08.

**Resumen rapido:** Gamerdust es una herramienta para construir y revisar narrativa de videojuegos. Permite crear proyectos narrativos, definir restricciones creativas, registrar lore, personajes, escenas y nodos de dialogo, y despues revisar validaciones narrativas, analisis emocional simulado y estimaciones de costo narrativo.

## 1. Stack y arquitectura general

La aplicacion combina backend Laravel con frontend React usando Inertia.

| Capa | Tecnologia | Archivos principales | Funcion |
|---|---|---|---|
| Backend | Laravel 13, PHP 8.3 | `app/`, `routes/web.php`, `database/` | Rutas, controladores, modelos, autorizacion, validacion, persistencia |
| Frontend | React 19 + TypeScript | `resources/js/` | Paginas, formularios y paneles de la interfaz |
| Puente Laravel/React | Inertia | `app/Http/Middleware/HandleInertiaRequests.php`, `resources/js/app.tsx` | Laravel renderiza paginas React sin API REST separada |
| Build frontend | Vite + Tailwind CSS | `vite.config.js`, `resources/css/app.css` | Compilacion y estilos |
| Persistencia | Base configurada por Laravel | `config/database.php`, migraciones | Tablas de usuarios y dominio narrativo |

No hay una API JSON separada en este estado. Las acciones de usuario se envian con formularios Inertia a rutas web de Laravel. Laravel valida, guarda, redirige y vuelve a renderizar la pagina React correspondiente con props actualizadas.

In [ ]:
# Celda opcional: inventario rapido del proyecto desde la raiz.
from pathlib import Path

root = Path.cwd()
interesting = [
    'routes/web.php',
    'app/Http/Controllers',
    'app/Models',
    'app/Services',
    'app/Http/Requests',
    'resources/js/pages',
    'resources/js/components/gamerdust',
    'database/migrations',
    'database/seeders',
]

for item in interesting:
    path = root / item
    print(f'\n{item}')
    if path.is_dir():
        for child in sorted(path.rglob('*')):
            if child.is_file():
                print('  -', child.relative_to(root))
    elif path.exists():
        print('  -', path.relative_to(root))
    else:
        print('  (no existe)')

## 2. Flujo general de usuario

El flujo previsto para una persona usuaria es:

1. Entrar a `/`. Si no hay sesion, Laravel muestra `Auth/Login`. Si hay sesion, redirige a `/projects`.
2. Registrarse o iniciar sesion.
3. Ver el listado de proyectos narrativos en `/projects`.
4. Crear un proyecto con titulo, genero, tipo narrativo, premisa, conflicto central, audiencia y estado.
5. Entrar al detalle del proyecto en `/projects/{project}`.
6. Definir restricciones creativas: tono emocional, emociones secundarias, tipo de mundo, rol del jugador, nivel de ramificacion, elementos requeridos/prohibidos y notas.
7. Crear piezas de contenido: lore, personajes, escenas y dialogos.
8. Revisar automaticamente advertencias narrativas en el panel de validaciones.
9. Ejecutar analisis emocional local sobre escenas o dialogos.
10. Revisar estimaciones de costo narrativo generadas para escenas y dialogos.

**Donde se revisa cada cosa desde la app:**

| Necesidad | Pantalla/ruta | Componente React |
|---|---|---|
| Login | `/login` | `resources/js/pages/Auth/Login.tsx` |
| Registro | `/register` | `resources/js/pages/Auth/Register.tsx` |
| Listado de proyectos | `/projects` | `resources/js/pages/Projects/Index.tsx` |
| Crear proyecto | `/projects/create` | `Projects/Create.tsx` + `ProjectForm.tsx` |
| Ver dashboard del proyecto | `/projects/{project}` | `Projects/Show.tsx` |
| Restricciones del proyecto | Dashboard del proyecto | `ConstraintForm.tsx` |
| Validaciones narrativas | Dashboard del proyecto | `NarrativeValidationPanel.tsx` |
| Analisis emocional | Dashboard del proyecto | `EmotionalAnalysisPanel.tsx` |
| Lore | `/projects/{project}/lore` | `Lore/Index.tsx`, `LoreEntryForm.tsx` |
| Personajes | `/projects/{project}/characters` | `Characters/Index.tsx`, `CharacterForm.tsx` |
| Escenas | `/projects/{project}/scenes` | `Scenes/Index.tsx`, `SceneForm.tsx` |
| Dialogos | `/projects/{project}/dialogues` | `Dialogues/Index.tsx`, `DialogueNodeForm.tsx` |

## 3. Rutas web

Las rutas principales estan en `routes/web.php`.

| Ruta | Metodo | Controlador/accion | Descripcion |
|---|---|---|---|
| `/` | GET | closure | Si hay sesion redirige a proyectos; si no, muestra login |
| `/login` | GET/POST | `AuthController@create/store` | Mostrar formulario e iniciar sesion |
| `/register` | GET/POST | `AuthController@register/storeRegistration` | Mostrar formulario y crear usuario |
| `/logout` | POST | `AuthController@destroy` | Cerrar sesion |
| `/projects` | resource | `ProjectController` | CRUD completo de proyectos |
| `/projects/{project}/constraints` | POST/PUT | `ProjectConstraintController@store` | Crear o actualizar restricciones |
| `/projects/{project}/lore` | resource sin show | `LoreEntryController` | CRUD de lore dentro de proyecto |
| `/projects/{project}/characters` | resource sin show | `CharacterController` | CRUD de personajes |
| `/projects/{project}/scenes` | resource sin show | `SceneController` | CRUD de escenas |
| `/projects/{project}/dialogues` | resource sin show | `DialogueNodeController` | CRUD de nodos de dialogo |
| `/projects/{project}/emotional-analysis` | GET/POST | `EmotionalAnalysisController` | Redirigir al dashboard o guardar analisis |
| `/projects/{project}/validations` | GET/POST/PATCH | `NarrativeValidationController` | Recalcular validaciones o marcarlas resueltas |
| `/projects/{project}/cost-estimations` | GET/POST | `CostEstimationController` | Redirigir al dashboard o guardar estimacion |

Todas las rutas de dominio estan protegidas con middleware `auth` y `verified`. La app marca usuarios registrados como verificados inmediatamente, asi que en la practica el flujo de email verification no esta expuesto al usuario.

## 4. Controladores

### `AuthController`

Gestiona login, registro y logout. Valida credenciales, usa `Auth::attempt`, regenera la sesion despues de autenticar y crea usuarios con password hasheado. En registro, tambien asigna `email_verified_at` con la fecha actual y autentica al usuario automaticamente.

### `ProjectController`

Controlador principal del dominio. Lista proyectos del usuario con conteos de lore, personajes, escenas y dialogos. Crea, muestra, edita, actualiza y elimina proyectos. En `show`, antes de cargar el proyecto, ejecuta `NarrativeValidationService->run($project)`, por lo que entrar al dashboard recalcula advertencias narrativas pendientes. Luego carga relaciones limitadas para el dashboard: restricciones, ultimas entradas de lore, personajes, escenas ordenadas, dialogos con escena/personaje, ultimo analisis emocional, validaciones no resueltas y estimaciones de costo.

### `ProjectConstraintController`

Guarda las restricciones creativas de un proyecto usando `updateOrCreate`. Hay una sola fila de restricciones por proyecto por la columna unica `project_id`.

### `LoreEntryController`

CRUD de entradas de lore. Cada entrada pertenece a un proyecto y se valida que el usuario tenga permiso de actualizar/ver el proyecto. Los metodos `edit`, `update` y `destroy` comprueban que el lore pertenece al proyecto recibido en la ruta.

### `CharacterController`

CRUD de personajes narrativos. Maneja datos de identidad, trasfondo, personalidad, motivacion, miedo, deseo, arco emocional, descripcion visual y notas de voz. Tambien protege que el personaje pertenezca al proyecto correcto.

### `SceneController`

CRUD de escenas. Al crear o actualizar una escena ejecuta `NarrativeCostEstimatorService->createFor($project, $scene)` y guarda el resultado en `cost_estimations`; ademas copia `cost_level` al campo `estimated_cost` de la escena para mostrarlo rapido en UI.

### `DialogueNodeController`

CRUD de nodos de dialogo. Al crear/editar carga listas de escenas, personajes y otros nodos para formularios. Al guardar sincroniza opciones: elimina las opciones anteriores y crea las nuevas. Tambien ejecuta la estimacion de costo sobre el nodo de dialogo.

### `EmotionalAnalysisController`

Permite elegir una fuente `scene` o `dialogue`, resolverla dentro del proyecto y crear un analisis emocional local. Para escenas usa `summary`; para dialogos usa `text`. Si no hay texto, aborta con codigo 422. El mensaje de exito dice explicitamente `Analisis emocional simulado guardado`.

### `NarrativeValidationController`

Permite recalcular validaciones y marcar una validacion como resuelta. Al recalcular, delega en `NarrativeValidationService`. Al resolver, actualiza `resolved` a `true`.

### `CostEstimationController`

Permite crear estimaciones de costo manualmente para escenas o dialogos desde una fuente seleccionada. Para escenas, tambien actualiza `estimated_cost` en la tabla `scenes`.

## 5. Modelos y relaciones

| Modelo | Tabla | Relaciones principales | Comentario |
|---|---|---|---|
| `User` | `users` | tiene muchos `Project` | Usuario propietario de proyectos |
| `Project` | `projects` | pertenece a `User`; tiene `constraint`, `loreEntries`, `characters`, `scenes`, `dialogueNodes`, `emotionalAnalyses`, `narrativeValidations`, `costEstimations`, `aiGenerations` | Nodo central de la app |
| `ProjectConstraint` | `project_constraints` | pertenece a `Project` | Restricciones creativas del proyecto |
| `LoreEntry` | `lore_entries` | pertenece a `Project`; tiene analisis emocionales polimorficos declarados | Piezas de mundo, historia, facciones, lugares, objetos, eventos o conceptos |
| `Character` | `characters` | pertenece a `Project`; tiene muchos `DialogueNode` | Personajes narrativos |
| `Scene` | `scenes` | pertenece a `Project`; tiene muchos `DialogueNode`; tiene muchos analisis/costos polimorficos | Escenas ordenadas del proyecto |
| `DialogueNode` | `dialogue_nodes` | pertenece a `Project`, `Scene`, `Character`; tiene `options`; tiene `incomingOptions`; tiene analisis/costos polimorficos | Unidad de dialogo, puede enlazarse con opciones |
| `DialogueOption` | `dialogue_options` | pertenece a `DialogueNode`; puede apuntar a `nextDialogueNode` | Opcion del jugador y salto opcional |
| `EmotionalAnalysis` | `emotional_analyses` | pertenece a `Project`; `morphTo analyzable` | Resultado de analisis emocional local |
| `NarrativeValidation` | `narrative_validations` | pertenece a `Project` | Advertencias narrativas pendientes o resueltas |
| `CostEstimation` | `cost_estimations` | pertenece a `Project`; `morphTo costable` | Estimacion de costo narrativo para escena/dialogo |
| `AiGeneration` | `ai_generations` | pertenece a `Project` y `User` | Tabla preparada para guardar generaciones IA, pero no se usa actualmente desde controladores/servicios |

Los modelos usan `$fillable` para controlar asignacion masiva. Algunos campos JSON se castean a arrays: emociones secundarias, elementos requeridos/prohibidos, puntajes emocionales, sugerencias y factores de costo.

## 6. Base de datos

La migracion principal de dominio es `database/migrations/2026_06_04_000000_create_gamerdust_tables.php`.

### `projects`

Representa un proyecto narrativo del usuario. Campos clave: `user_id`, `title`, `description`, `game_genre`, `narrative_type`, `premise`, `central_conflict`, `target_audience`, `status`, timestamps. `status` tiene indice. Si se borra el usuario, se borran sus proyectos por `cascadeOnDelete`.

### `project_constraints`

Una fila por proyecto gracias a `project_id` unico. Guarda tono emocional, emocion dominante, emociones secundarias JSON, tipo de mundo, rol del jugador, ramificacion, elementos requeridos/prohibidos JSON y notas creativas.

### `lore_entries`

Guarda piezas de lore. Campos: `project_id`, `title`, `type`, `description`, `narrative_function`, `emotional_tone`. `type` tiene indice.

### `characters`

Guarda personajes. Campos: nombre, rol, arquetipo, edad, genero, trasfondo, personalidad, estilo de habla, motivacion, miedo, deseo, arco emocional, descripcion visual y notas de voz.

### `scenes`

Guarda escenas. Campos: titulo, orden, resumen, conflicto, emocion esperada, nivel de interes, tipo, decision del jugador, consecuencia y costo estimado. Tiene indice compuesto `project_id` + `order` para ordenar escenas de un proyecto.

### `dialogue_nodes`

Guarda nodos de dialogo. Puede asociarse a escena y personaje. `node_key` es unico dentro de cada proyecto con indice unico `project_id` + `node_key`. Incluye tipo de dialogo, texto, tono emocional y condicion de disparo.

### `dialogue_options`

Guarda opciones del jugador para un nodo. Cada opcion puede tener texto, consecuencia, condicion requerida y un salto a otro nodo por `next_dialogue_node_id`.

### `emotional_analyses`

Guarda resultados de analisis emocional. Usa relacion polimorfica `analyzable`, asi que puede apuntar a distintos tipos de contenido. En la practica actual el controlador permite analizar escenas y dialogos. Guarda texto original, emocion dominante, valence/arousal/dominance, puntajes JSON, interpretacion y sugerencias JSON.

### `narrative_validations`

Guarda advertencias de coherencia narrativa. Campos: tipo de validacion, severidad, mensaje, recomendacion y `resolved`. Tiene indices para tipo, severidad y resuelto.

### `cost_estimations`

Guarda costo narrativo calculado. Usa relacion polimorfica `costable`, actualmente para escenas y dialogos. Incluye nivel (`low`, `medium`, `high`), puntaje numerico, factores JSON y explicacion.

### `ai_generations`

Tabla preparada para guardar generaciones de IA: proyecto, usuario, tipo de generacion, prompt, response JSON y modelo. Actualmente no hay controlador ni servicio que la escriba.

In [ ]:
# Celda opcional: inspeccionar un SQLite local si el proyecto usa database/database.sqlite.
# Si tu .env apunta a MySQL/MariaDB, esta celda solo avisara que no encontro SQLite.
from pathlib import Path
import sqlite3

db_path = Path.cwd() / 'database' / 'database.sqlite'
if not db_path.exists():
    print('No existe database/database.sqlite. Revisa .env para ver el motor de base de datos configurado.')
else:
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    tables = [row[0] for row in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    for table in tables:
        print(f'\n[{table}]')
        for col in cur.execute(f'PRAGMA table_info({table})'):
            print(col)
    con.close()

## 7. Validaciones de entrada

La app usa `FormRequest` para reglas de datos.

| Request | Uso | Reglas destacadas |
|---|---|---|
| `StoreProjectRequest` | Crear proyecto | `title` obligatorio; `narrative_type` debe ser `game_story`, `balanced_narrative` o `player_story`; `status` debe ser `draft`, `in_progress` o `completed` |
| `UpdateProjectRequest` | Actualizar proyecto | Mismas reglas que crear, pero autoriza con policy sobre el proyecto |
| `StoreProjectConstraintRequest` | Guardar restricciones | Convierte strings separados por comas/saltos en arrays para emociones y elementos; valida arrays y textos |
| `StoreLoreEntryRequest` | Crear/editar lore | `title` requerido; `type` debe ser `rule`, `history`, `faction`, `location`, `object`, `event` o `concept` |
| `StoreCharacterRequest` | Crear/editar personaje | `name` requerido; edad entera entre 0 y 200; textos opcionales para motivacion, miedo, deseo, etc. |
| `StoreSceneRequest` | Crear/editar escena | `title` requerido; `order` entero minimo 1; `interest_level` entre 1 y 10 |
| `StoreDialogueNodeRequest` | Crear/editar dialogo | `node_key` requerido y unico por proyecto; `scene_id`/`character_id` deben pertenecer al proyecto; `dialogue_type` limitado a tipos declarados; opciones opcionales con salto a nodo valido |

La autorizacion se apoya en `ProjectPolicy`: un usuario solo puede ver, actualizar o eliminar proyectos que le pertenecen.

## 8. Servicios de dominio

### `NarrativeValidationService`

Ejecuta reglas simples de coherencia. Antes de recalcular, borra las validaciones no resueltas del proyecto para no duplicarlas. Luego crea hallazgos si detecta:

- Proyecto sin premisa: severidad `high`.
- Personaje sin motivacion: severidad `medium`.
- Escena sin conflicto: severidad `medium`.
- Lore sin funcion narrativa: severidad `low`.

Estas advertencias se revisan en el dashboard del proyecto, dentro de `NarrativeValidationPanel.tsx`. El usuario puede pulsar `Recalcular` o `Resolver`.

### `EmotionalAnalysisService`

No llama a un proveedor de IA. Normaliza el texto a minusculas y puntua emociones mediante palabras clave en ingles y espanol. Emociones consideradas: `joy`, `fear`, `anger`, `sadness`, `tension`. La emocion dominante es la de mayor puntaje. Tambien calcula valores simulados de `valence`, `arousal` y `dominance`, y guarda sugerencias genericas.

Palabras clave de ejemplo:

- `joy`: hope, joy, victory, friend, celebration, luz.
- `fear`: fear, terror, dark, threat, monster, miedo.
- `anger`: rage, betrayal, revenge, war, furia, ira.
- `sadness`: loss, grief, alone, death, tristeza, duelo.
- `tension`: conflict, choice, risk, secret, conflicto, riesgo.

### `NarrativeCostEstimatorService`

Calcula costo narrativo con reglas locales. Para escenas, sube el puntaje si hay decision del jugador, consecuencia y cuatro o mas dialogos asociados. Para dialogos, parte de costo bajo si el tipo es `bark` o `ambient`; otros tipos arrancan en medio. Si hay mas de una opcion, suma ramificacion. El resultado se guarda en `cost_estimations` y, para escenas, tambien en `scenes.estimated_cost`.

## 9. IA: estado actual

**Conclusion:** la aplicacion todavia no tiene IA real integrada.

Evidencias en el codigo:

- `EmotionalAnalysisController` devuelve el mensaje `Analisis emocional simulado guardado`.
- `EmotionalAnalysisPanel.tsx` indica que es un `Placeholder local con palabras clave, listo para conectar IA despues`.
- `EmotionalAnalysisService` usa `Str::contains` y listas de palabras clave; no usa OpenAI, Anthropic, Hugging Face, Ollama ni ningun cliente HTTP.
- `NarrativeValidationService` usa reglas deterministas simples, no IA.
- `NarrativeCostEstimatorService` usa heuristicas deterministas, no IA.
- Existe el modelo/tabla `AiGeneration`, pero actualmente no se observa uso desde rutas, controladores o servicios.

Por tanto, la app esta preparada conceptualmente para funciones asistidas por IA, especialmente por la tabla `ai_generations`, pero hasta ahora lo que hay es simulacion local y reglas de negocio.

## 10. Frontend: paginas y componentes

### Entrada React/Inertia

`resources/js/app.tsx` inicializa Inertia, resuelve paginas desde `resources/js/pages/**/*.tsx`, monta React con `createRoot` y configura la barra de progreso color amber.

### Layout

`AppShell.tsx` es la carcasa visual principal para paginas autenticadas. Muestra cabecera, nombre Gamerdust, enlace a proyectos, usuario autenticado y mensajes flash de exito/error.

### Paginas de autenticacion

- `Auth/Login.tsx`: formulario de email/password, recordar sesion y enlace a registro.
- `Auth/Register.tsx`: formulario de nombre, email, password y confirmacion.

### Paginas de proyectos

- `Projects/Index.tsx`: lista proyectos paginados y usa `ProjectCard`.
- `Projects/Create.tsx`: muestra `ProjectForm` para crear.
- `Projects/Edit.tsx`: muestra `ProjectForm` para editar.
- `Projects/Show.tsx`: dashboard central del proyecto. Muestra resumen, estadisticas, restricciones, validaciones, analisis emocional, accesos rapidos y secciones recientes de lore, personajes, escenas y dialogos.

### Paginas CRUD de contenido

- `Lore/Index.tsx`, `Lore/Create.tsx`, `Lore/Edit.tsx`: listar, crear y editar lore.
- `Characters/Index.tsx`, `Characters/Create.tsx`, `Characters/Edit.tsx`: listar, crear y editar personajes.
- `Scenes/Index.tsx`, `Scenes/Create.tsx`, `Scenes/Edit.tsx`: listar, crear y editar escenas.
- `Dialogues/Index.tsx`, `Dialogues/Create.tsx`, `Dialogues/Edit.tsx`: listar, crear y editar nodos de dialogo.

### Componentes de dominio

| Componente | Funcion |
|---|---|
| `ProjectForm` | Formulario de proyecto: titulo, descripcion, genero, tipo narrativo, estado, premisa, conflicto, audiencia |
| `ProjectCard` | Tarjeta/resumen de proyecto en listado |
| `ProjectStats` | Muestra conteos de lore, personajes, escenas y dialogos |
| `ConstraintForm` | Edita restricciones creativas del proyecto |
| `NarrativeValidationPanel` | Lista advertencias, permite recalcular y resolver |
| `EmotionalAnalysisPanel` | Permite elegir escena/dialogo y ejecutar analisis emocional local |
| `CostEstimationBadge` | Muestra etiqueta visual de costo bajo/medio/alto |
| `LoreEntryForm` | Formulario de lore |
| `LoreEntryCard` | Tarjeta de lore |
| `CharacterForm` | Formulario de personaje |
| `CharacterCard` | Tarjeta de personaje |
| `SceneForm` | Formulario de escena |
| `SceneCard` | Tarjeta de escena, incluye costo estimado si aplica |
| `DialogueNodeForm` | Formulario de nodo de dialogo con opciones del jugador y salto a otro nodo |
| `DialogueNodeCard` | Tarjeta de nodo de dialogo con escena/personaje/opciones |

Los formularios usan `useForm` de Inertia. Eso significa que mantienen estado local en React, envian `post` o `put`, reciben errores de validacion desde Laravel y muestran feedback sin construir manualmente una API REST.

## 11. Seed inicial

`database/seeders/DatabaseSeeder.php` crea datos de ejemplo para probar el flujo completo:

- Usuario: `test@example.com` con password `password`.
- Proyecto: `Dustfall Protocol`.
- Restricciones: tono melancolico esperanzado, emocion dominante tension, mundo de colonia aislada, rol archivista, ramificacion moderada.
- Lore: `El Polvo de Vigilia`.
- Personaje: `Mara Venn`.
- Escena: `Archivo bajo ceniza`.
- Dialogo: `archive_intro_001` con una opcion de jugador.

Este seed permite entrar, revisar el dashboard y probar analisis, validaciones y costos sin crear datos desde cero.

## 12. Seguridad y autorizacion

La seguridad actual se basa en:

- Middleware `auth` y `verified` para rutas de dominio.
- `ProjectPolicy` para controlar acceso por propietario.
- Comprobaciones manuales `abort_unless(..., 404)` en recursos hijos para evitar acceder a personajes, escenas, lore o dialogos de otro proyecto.
- Validaciones con `FormRequest` para entradas persistentes.
- `Hash::make` para passwords.
- Regeneracion de sesion tras login y registro.

Punto a vigilar: como los usuarios se marcan verificados automaticamente al registrarse, `verified` no representa una verificacion real por email en este estado.

## 13. Ciclos principales de datos

### Crear proyecto

`ProjectForm` envia datos a `ProjectController@store`. `StoreProjectRequest` valida. El proyecto se crea desde `$request->user()->projects()`. Laravel redirige al dashboard.

### Ver dashboard

`ProjectController@show` autoriza el proyecto, recalcula validaciones narrativas y carga relaciones resumidas. `Projects/Show.tsx` reparte la informacion a paneles y tarjetas.

### Crear escena

`SceneForm` envia a `SceneController@store`. Se crea la escena, se calcula costo con `NarrativeCostEstimatorService`, se guarda una fila en `cost_estimations` y se actualiza `scenes.estimated_cost`.

### Crear dialogo

`DialogueNodeForm` envia nodo y opciones. `DialogueNodeController` crea el nodo, borra/sincroniza opciones, calcula costo y redirige al listado de dialogos.

### Analizar emocion

`EmotionalAnalysisPanel` envia `source_type` y `source_id`. El controlador resuelve la escena/dialogo dentro del proyecto, toma el texto, ejecuta `EmotionalAnalysisService` y guarda `emotional_analyses`.

### Validar narrativa

El panel de validaciones ejecuta POST a `/projects/{project}/validations`. El servicio borra validaciones no resueltas y recrea advertencias segun reglas simples. Resolver una advertencia hace PATCH y marca `resolved = true`.

In [ ]:
# Celda opcional: buscar referencias a IA/generacion/modelos externos en el codigo.
from pathlib import Path
import re

terms = re.compile(r'OpenAI|Anthropic|Hugging|Ollama|ai_generation|AiGeneration|model|prompt|response|simulado|placeholder', re.IGNORECASE)
for path in Path.cwd().rglob('*'):
    if not path.is_file():
        continue
    if any(part in {'.git', 'node_modules', 'vendor', 'storage'} for part in path.parts):
        continue
    if path.suffix.lower() not in {'.php', '.tsx', '.ts', '.js', '.json'}:
        continue
    text = path.read_text(encoding='utf-8', errors='ignore')
    if terms.search(text):
        print(path.relative_to(Path.cwd()))

## 14. Como correr y revisar la app

Comandos utiles desde la raiz:

```bash
composer install
npm install
php artisan migrate --seed
composer run dev
```

Con el seed, se puede entrar con:

- Email: `test@example.com`
- Password: `password`

Para pruebas automatizadas, `composer test` ejecuta `php artisan test`. En el estado revisado solo aparecen tests de ejemplo (`tests/Feature/ExampleTest.php`, `tests/Unit/ExampleTest.php`), por lo que todavia falta cobertura especifica del dominio Gamerdust.

## 15. Estado actual y siguientes pasos sugeridos

La app ya tiene una base funcional para gestionar narrativa de videojuegos: autenticacion, proyectos, restricciones, lore, personajes, escenas, dialogos, validaciones, analisis emocional simulado y costos narrativos.

Lo que todavia no tiene:

- IA real conectada a un modelo externo.
- Uso efectivo de la tabla `ai_generations`.
- API dedicada para integraciones externas.
- Tests de dominio para controladores, policies, requests y servicios.
- Validaciones narrativas avanzadas con contexto cruzado profundo.

Posibles siguientes pasos tecnicos:

1. Conectar un proveedor de IA para analisis/generacion y guardar prompts/respuestas en `ai_generations`.
2. Ampliar `NarrativeValidationService` con reglas de continuidad, contradicciones, arcos emocionales y dependencia entre escenas.
3. Agregar tests para ownership de proyectos y recursos hijos.
4. Crear tests unitarios de los servicios de costo, validacion y analisis emocional.
5. Mejorar UI de grafo de dialogos para visualizar saltos entre nodos.
6. Agregar filtros/busqueda en lore, personajes, escenas y dialogos.